# get rid of  "Halo" ? | cleaner post-preprocessing
(roundish dorsal motor cortex-anchoring?!)
--> Clean TS differently to remove weird gradient patterns - e.g.


### Variants
1. Different confounds: classic = 36 P
2. above + scrubbing (FD < 0.3)
3. above + run-removal without at least 240 sec (104 frames) of FD < 0.3

### add
tried xcp_d, but a bit complicated with required input (outpur file format from fmriprep - want space-fsLR & cifit files)
should theorectically do very similar steps as nilearn.signal.clean + scrubbing etc. (?!)

In [45]:
import os.path as op
import numpy as np
import os 

bids_folder_base = '/mnt_03/ds-dnumrisk'
bids_folder_out = '/mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-dnumrisk'
task = 'magjudge'

sub = '06'
ses = 1

In [ ]:
# original confounds
fmriprep_confounds_include = ['global_signal', 'dvars', 'framewise_displacement', 'trans_x',
                                'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z',
                                'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03', 'cosine00', 'cosine01', 'cosine02']



In [ ]:
# As Katerina suggested:
# You should definitely try adding the derivative1 and power2 of the trans and rot variables you already have (and maybe even the derivative1_power2 of these variables as well).
confspec = 'kate1'
fmriprep_confounds_include = ['dvars', 'framewise_displacement', 
                            'trans_x','trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z',
                            'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03',
                            'cosine00', 'cosine01', 'cosine02']

# add derivative1 and power2 of the trans and rot variables 
mov_params = ['trans_x','trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
for param in mov_params:
    fmriprep_confounds_include.append(param + '_derivative1')
    fmriprep_confounds_include.append(param + '_power2')
    fmriprep_confounds_include.append(param + '_derivative1_power2')


In [ ]:
confspec = 'kate2' # same as 36P which I use in getCM script
# you seem to have done the acompcor one but other common strategies don't include acompcor but white matter and CSF

mov_params = ['trans_x','trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
general_params = ['csf','white_matter','global_signal']
base_params = mov_params + general_params

fmriprep_confounds_include = base_params.copy()
for param in base_params: # add derivative1 and power2 to all realignment parameters and others 
    fmriprep_confounds_include.append(param + '_derivative1')
    fmriprep_confounds_include.append(param + '_power2')
    fmriprep_confounds_include.append(param + '_derivative1_power2')


In [36]:
# scubbing
# needs sample_mask 

scrubbing = True
scrub_threshold = 0.3
TR = 2.3 # forgot to set it before, default is 2.5

confspec += '_scrub'

In [62]:
import nibabel as nib
import pandas as pd
from nilearn import signal


def cleanTS(sub, ses =1, task ='magjudge',runs = range(1, 7),space = 'fsaverage5', bids_folder='/Users/mrenke/data/ds-dnumrisk', 
        fmriprep_confounds_include = fmriprep_confounds_include, scrubbing=False): #  'magjudge'
    print(fmriprep_confounds_include)
    fmriprep_folder = op.join(bids_folder,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func') # f'ses-{ses}', 

    number_of_vertices = 20484
    clean_ts_runs = np.empty([number_of_vertices,0])
    for run in runs:
        timeseries = [None] * 2
        for i, hemi in enumerate(['L', 'R']):  
            filename =  op.join(fmriprep_folder, f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_space-{space}_hemi-{hemi}_bold.func.gii')   #_ses-{ses}
            timeseries[i] = nib.load(filename).agg_data()        
        timeseries = np.vstack(timeseries) # (20484, N_timepoints)
        # load in and remove confounds
        fmriprep_confounds_file = op.join(fmriprep_folder,f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_desc-confounds_timeseries.tsv') # _ses-{ses} timeseries
        fmriprep_confounds = pd.read_table(fmriprep_confounds_file)[fmriprep_confounds_include] 
        fmriprep_confounds= fmriprep_confounds.bfill()

        if scrubbing:
            print('performing scrubbing with threshold', scrub_threshold)
            sample_mask = (pd.read_table(fmriprep_confounds_file)['framewise_displacement'] < scrub_threshold).to_numpy()
            print(f'removed {np.sum(sample_mask == False)} frames')
            clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T
        else:
            clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, standardize='zscore_sample').T
        clean_ts_runs = np.append(clean_ts_runs, clean_ts, axis=1)
    return clean_ts_runs


In [81]:
sub = '02'
clean_ts = cleanTS(sub, bids_folder=bids_folder_base, scrubbing=scrubbing, fmriprep_confounds_include=fmriprep_confounds_include)

['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'csf', 'white_matter', 'global_signal', 'trans_x_derivative1', 'trans_x_power2', 'trans_x_derivative1_power2', 'trans_y_derivative1', 'trans_y_power2', 'trans_y_derivative1_power2', 'trans_z_derivative1', 'trans_z_power2', 'trans_z_derivative1_power2', 'rot_x_derivative1', 'rot_x_power2', 'rot_x_derivative1_power2', 'rot_y_derivative1', 'rot_y_power2', 'rot_y_derivative1_power2', 'rot_z_derivative1', 'rot_z_power2', 'rot_z_derivative1_power2', 'csf_derivative1', 'csf_power2', 'csf_derivative1_power2', 'white_matter_derivative1', 'white_matter_power2', 'white_matter_derivative1_power2', 'global_signal_derivative1', 'global_signal_power2', 'global_signal_derivative1_power2']
performing scrubbing with threshold 0.3
removed 41 frames


/tmp/ipykernel_1028739/1896481491.py:28: DeprecationWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.13, the default strategy will be replaced by the new strategy and the 'zscore' option will be removed. Please use 'zscore_sample' instead.
  clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T


performing scrubbing with threshold 0.3
removed 89 frames


/tmp/ipykernel_1028739/1896481491.py:28: DeprecationWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.13, the default strategy will be replaced by the new strategy and the 'zscore' option will be removed. Please use 'zscore_sample' instead.
  clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T


performing scrubbing with threshold 0.3
removed 44 frames


...

/tmp/ipykernel_1028739/1896481491.py:28: DeprecationWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.13, the default strategy will be replaced by the new strategy and the 'zscore' option will be removed. Please use 'zscore_sample' instead.
  clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T


performing scrubbing with threshold 0.3
removed 157 frames


/tmp/ipykernel_1028739/1896481491.py:28: DeprecationWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.13, the default strategy will be replaced by the new strategy and the 'zscore' option will be removed. Please use 'zscore_sample' instead.
  clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T


performing scrubbing with threshold 0.3
removed 158 frames


/tmp/ipykernel_1028739/1896481491.py:28: DeprecationWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.13, the default strategy will be replaced by the new strategy and the 'zscore' option will be removed. Please use 'zscore_sample' instead.
  clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T


In [82]:
from utils import get_basic_mask

mask, labeling_noParcel = get_basic_mask()
seed_ts = clean_ts[mask]

from nilearn.connectome import ConnectivityMeasure
correlation_measure = ConnectivityMeasure(kind='correlation')
cm = correlation_measure.fit_transform([seed_ts.T])[0] #correlation_matrix_noParcel
print(f'sub-{sub} ses-{ses} task-{task}: raw connectivity matrix estimated') 
target_fn =  op.join(bids_folder_out, 'derivatives', 'correlation_matrices.tryNoHalo', f'sub-{sub}_ses-1_task-magjudge_confspec-{confspec}_CM-unfiltered.npy')
np.save(target_fn, cm)
print(f'saving to {target_fn}')

[get_dataset_dir] Dataset found in /home/ubuntu/nilearn_data/destrieux_surface
sub-02 ses-1 task-magjudge: raw connectivity matrix estimated
saving to /mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-dnumrisk/derivatives/correlation_matrices.tryNoHalo/sub-02_ses-1_task-magjudge_confspec-kate2_scrub_CM-unfiltered.npy


In [83]:
target_dir = op.join(bids_folder_out, 'derivatives', 'gradients.tryNoHalo', f'sub-{sub}', f'ses-1')
if not op.exists(target_dir):
    os.makedirs(target_dir)

bids_folder_ref = bids_folder_base
specification = confspec

# filter out nodes that are not connected to the rest
from scipy.sparse.csgraph import connected_components
cc_mask_file = op.join(target_dir,f'sub-{sub}_ses-{ses}_task-{task}_cc-mask_space-fsaverag5.npy')
if (os.path.exists(cc_mask_file) == False):
    cc = connected_components(cm)
    mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]
    np.save(cc_mask_file, mask_cc) # save all together
    print('connected components derived & mask saved')    
mask_cc = np.load(cc_mask_file)
mask, labeling_noParcel = get_basic_mask()
mask[mask == True] = mask_cc # mark nodes not in component 0  as False in mask
cm_filtered = cm[mask_cc, :][:, mask_cc]
print('connectivty matrix loaded and filtered with cc_mask')    

# load in reference gradient and apply same filter
g_ref = np.load(op.join(bids_folder_ref,'derivatives', 'gradients','sub-All', 'sub-All_gradients_N-10.npy')) # 
print(np.shape(g_ref))
g_ref_fil = g_ref[:,mask] # np.shape(g_ref) = (10,20484)

connected components derived & mask saved
[get_dataset_dir] Dataset found in /home/ubuntu/nilearn_data/destrieux_surface
connectivty matrix loaded and filtered with cc_mask
(10, 20484)


In [84]:
from brainspace.gradient import GradientMaps
from brainspace.utils.parcellation import map_to_labels
n_components = 10
# now perform embedding on cleaned data + alignment
print(f'start fitting gradients now')
gm = GradientMaps(n_components=n_components,alignment='procrustes') # defaults: approacch = 'dm', kernel = None
gm.fit(cm_filtered,reference=g_ref_fil.T)
print(f'gradients generated')

# save results
np.save(op.join(target_dir,f'sub-{sub}_ses-{ses}_task-{task}_lambdas_space-fsaverag5_n10_{specification}.npy'), gm.lambdas_) # save all together
gm_= gm.gradients_.T 
grad = [None] * n_components
for i, g in enumerate(gm_): # gm.gradients_.T
    grad[i] = map_to_labels(g, labeling_noParcel, mask=mask, fill=np.nan)
np.save(op.join(target_dir,f'sub-{sub}_ses-{ses}_task-{task}_gradients_space-fsaverag5_n10_{specification}.npy'), grad) # save all together
gm_ = gm.aligned_.T
grad = [None] * n_components
for i, g in enumerate(gm_): # gm.gradients_.T
    grad[i] = map_to_labels(g, labeling_noParcel, mask=mask, fill=np.nan)
np.save(op.join(target_dir,f'sub-{sub}_ses-{ses}_task-{task}_g-aligned_space-fsaverag5_n10_{specification}.npy'), grad) # save all together    
print(f'finished sub-{sub} ses-{ses} task-{task} conf-{confspec}: gradients saved')

start fitting gradients now


/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/brainspace-0.1.4-py3.10.egg/brainspace/gradient/embedding.py:70: UserWarning: Affinity is not symmetric. Making symmetric.
  warnings.warn('Affinity is not symmetric. Making symmetric.')


gradients generated
finished sub-02 ses-1 task-magjudge conf-kate2_scrub: gradients saved


In [ ]:
from utils import plot_grads
sub = '02'
target_dir = op.join(bids_folder_out, 'derivatives', 'gradients.tryNoHalo', f'sub-{sub}', f'ses-1')
spec = 'gradients' # 'g-aligned'

# new gradients (different confounds)
confspec = 'kate2_scrub' # 'kate2'
grad = np.load(op.join(target_dir,f'sub-{sub}_ses-{ses}_task-{task}_{spec}_space-fsaverag5_n10_{confspec}.npy'))
plot_grads(grad, sub, spec, confspec)

# new gradients (different confounds)
confspec = 'kate2' # 'kate2'
grad = np.load(op.join(target_dir,f'sub-{sub}_ses-{ses}_task-{task}_{spec}_space-fsaverag5_n10_{confspec}.npy'))
plot_grads(grad, sub, spec, confspec)

# old gradient
confspec = 'old-confounds'
grad = np.load(op.join(bids_folder_base, 'derivatives', 'gradients', f'sub-{sub}' , f'sub-{sub}_{spec}_space-fsaverag5_n10.npy')) 
plot_grads(grad, sub, spec, confspec)

In [ ]:
# plotting gradients

import nilearn.plotting as nplt
import matplotlib.pyplot as plt
from  nilearn.datasets import fetch_surf_fsaverage
fsaverage = fetch_surf_fsaverage() # default 5

def plot_grads(grad, sub, spec, confspec):
    side_view = 'medial'
    cmap = 'jet'
    n_comp = 5

    figure, axes = plt.subplots(nrows=1, ncols=n_comp,figsize = (15,4), subplot_kw=dict(projection='3d'))
    for i in range(0,n_comp):
        gm = np.split(grad[i],2) # for i, hemi in enumerate([‘L’, ‘R’]): --> left first
        gm_r = gm[1]
        nplt.plot_surf(surf_mesh= fsaverage.infl_right, surf_map= gm_r, # infl_right # pial_right
                    view= side_view,cmap=cmap, colorbar=False,  # sub-{sub}, title=f’grad {i+1}‘,
                    bg_map=fsaverage.sulc_right, bg_on_data=True,darkness=0.7 ,axes=axes[i]) #
        axes[i].set(title=f'grad {i+1}')
    figure.suptitle(f'sub-{sub} {spec} {confspec}', y=0.9)

### Remove runs/subjects with excessive motion

Bianca: `what we do is that we have set to exclude runs (because each subject has multiple runs of rs-fMRI, which get treated independently) if there is less than 240s worth of data after censoring of high motion volumes (using fd threshold of 0.3, which I think is pretty standard) pretty standard as in also used in adults I think, and children move more, so we do end up excluding quite a few runs. But our final QC decision is about only keeping subjects that having minimum 2 runs (so minimum 2x 240s = 8min of data)`

In [97]:
TR = 2.3
vols_per_run = 188

(vols_per_run * TR)/60

7.206666666666666

In [103]:
# 2at least 40 sec per run means N-volumes:
240/TR

104.34782608695653

In [98]:
vols_per_run * TR

432.4

In [89]:
sub = '02'
ses = 1
runs = range(1, 7)
space = 'fsaverage5'
task = 'magjudge'

In [104]:
vol_per_run_thresh = 104
run_per_sub_thresh = 2

In [ ]:
fmriprep_folder = op.join(bids_folder_base,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func') # f'ses-{ses}', 

number_of_vertices = 20484
clean_ts_runs = np.empty([number_of_vertices,0])
N_valid_runs = 0
for run in runs:
    timeseries = [None] * 2
    for i, hemi in enumerate(['L', 'R']):  
        filename =  op.join(fmriprep_folder, f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_space-{space}_hemi-{hemi}_bold.func.gii')   #_ses-{ses}
        timeseries[i] = nib.load(filename).agg_data()  
    fmriprep_confounds_file = op.join(fmriprep_folder,f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_desc-confounds_timeseries.tsv') # _ses-{ses} timeseries
    fmriprep_confounds = pd.read_table(fmriprep_confounds_file)[fmriprep_confounds_include] 
    fmriprep_confounds= fmriprep_confounds.bfill()
    sample_mask = (pd.read_table(fmriprep_confounds_file)['framewise_displacement'] < scrub_threshold).to_numpy()
    usable_frames = np.sum(sample_mask == True)

    if usable_frames > vol_per_run_thresh:
        print(f'run {run} has {usable_frames} usable frames')
        clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds, sample_mask=sample_mask, t_r = TR, standardize='zscore_sample').T
        clean_ts_runs = np.append(clean_ts_runs, clean_ts, axis=1)
        N_valid_runs += 1
    else:
        print(f'run {run} has {usable_frames} usable frames, not usable')
if N_valid_runs < run_per_sub_thresh:
    print(f'sub-{sub} has {N_valid_runs} valid runs, not usable')
else:   


performing scrubbing with threshold 0.3
usable frames: 147
removed 41 frames
performing scrubbing with threshold 0.3
usable frames: 99
removed 89 frames
performing scrubbing with threshold 0.3
usable frames: 144
removed 44 frames
performing scrubbing with threshold 0.3
usable frames: 102
removed 86 frames
performing scrubbing with threshold 0.3
usable frames: 31
removed 157 frames
performing scrubbing with threshold 0.3
usable frames: 30
removed 158 frames


In [94]:
np.array(timeseries).shape

(2, 10242, 188)

In [96]:
(188 * TR)/60

7.206666666666666

In [88]:
fd_info = pd.read_csv(op.join(bids_folder_base,'add_tables','QC_framewise_displacement.csv'))
fd_info

,subject,run_1_mean,run_1_max,run_2_mean,run_2_max,run_3_mean,run_3_max,run_4_mean,run_4_max,run_5_mean,run_5_max,run_6_mean,run_6_max,overall_mean,overall_max
0,1,0.090440,0.281370,0.066592,0.170196,0.080448,0.355698,0.074131,0.227716,0.077681,0.292386,0.104021,0.773484,0.082219,0.773484
1,2,0.247483,1.128050,0.341725,2.012009,0.232820,0.824362,0.342942,1.730444,0.951775,7.740737,0.715813,2.670990,0.472093,7.740737
2,3,0.066980,0.495146,0.169703,2.486926,0.232428,2.023877,0.284133,3.262951,0.224385,1.903143,0.271422,1.534335,0.208175,3.262951
3,4,0.118423,0.380943,0.138568,0.523805,0.144430,0.666036,0.152893,0.979707,0.146906,0.564822,0.127962,0.409166,0.138197,0.979707
4,5,0.087899,0.430041,0.079283,0.402489,0.088875,0.537079,0.126875,0.863824,NaN,NaN,0.116834,0.706099,0.099953,0.863824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,62,0.069380,0.196827,0.068869,0.249237,NaN,NaN,0.066053,0.146038,0.084856,0.608301,0.072640,0.374305,0.072359,0.608301
62,63,0.118448,0.761462,0.268346,7.587189,0.270062,4.664563,0.154891,3.221470,0.156213,4.207050,0.107293,2.016423,0.179209,7.587189
63,64,0.110533,0.604411,0.144806,1.068267,0.138437,0.683087,0.156364,0.703755,0.158321,0.688715,0.160660,1.162581,0.144854,1.162581
64,65,0.093991,0.241274,0.074754,0.159199,0.080606,0.247617,0.109902,0.324036,0.093220,0.337020,0.125546,0.377501,0.096336,0.377501
